# 3) Reconstruct Target File with Source Collection Frames

This notebook performs **audio mosaicing**: each frame of a target audio is replaced by the most similar frame from the source collection, using KNN search over audio features.

**Experiments in this notebook:**

| # | Experiment | Variable | Target |
|---|---|---|---|
| 1 | Baseline | MFCCs only | Kick + Acid |
| 2 | Feature set comparison | 4 feature combinations | Kick |
| 3 | Randomization | Deterministic vs top-3 vs top-10 | Kick |
| 4 | Frame size | 2048 / 4096 / 8192 samples | Kick |
| 5 | Beat-sync segmentation | Fixed vs beat-aligned frames | Kick |

## 1. Setup

In [ ]:
%pip install essentia -q
%pip install git+https://github.com/mtg/freesound-python.git -q

In [ ]:
%cd /content/audio-mosaicing

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import essentia
import essentia.standard as estd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.neighbors import NearestNeighbors
from IPython.display import display, Audio

# ── Color palette ──────────────────────────────────────────────────────────────
C = {
    'deep_purple'  : '#3B1055',
    'mid_purple'   : '#6B2F9B',
    'light_purple' : '#9D6BBF',
    'accent'       : '#C084FC',
    'dark_grey'    : '#2E2E3A',
    'mid_grey'     : '#6B6B7E',
    'light_grey'   : '#B0B0C4',
    # per-experiment gradient (dark → light)
    'exp'          : ['#3B1055', '#6B2F9B', '#9D6BBF', '#6B6B7E', '#B0B0C4'],
}

plt.rcParams.update({
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.15,
    'grid.color'        : '#888888',
    'font.size'         : 11,
})

print("Setup complete. Essentia", essentia.__version__)

## 2. Load DataFrames

In [ ]:
df        = pd.read_csv('dataframe.csv',        index_col=0)
df_source = pd.read_csv('dataframe_source.csv', index_col=0)

print(f"Source metadata : {len(df)} sounds")
print(f"Source analysis : {len(df_source)} frames")
print(f"Source features : {list(df_source.columns[:8])} ...")

## 3. Core Functions

The main function is `reconstruct()` — a clean wrapper around the mosaicing algorithm that accepts any feature set and randomization level. This makes running all experiments straightforward.

In [ ]:
loaded_audio_files = {}

def get_audio_file_segment(file_path, start_sample, n_samples):
    """Load an audio segment from disk (with in-memory caching)."""
    if file_path not in loaded_audio_files:
        loaded_audio_files[file_path] = estd.MonoLoader(filename=file_path)()
    audio = loaded_audio_files[file_path]
    return audio[int(start_sample):int(start_sample) + int(n_samples)]


def reconstruct(df_target, df_source, feature_set, n_random=1, label='exp'):
    """
    Run audio mosaicing reconstruction.

    Parameters
    ----------
    df_target    : DataFrame — target file analysis
    df_source    : DataFrame — source collection analysis
    feature_set  : list[str] — features to use for KNN similarity
    n_random     : int — pick randomly from top-N neighbors (1 = deterministic)
    label        : str — used for output filename and plot titles

    Returns
    -------
    dict with audio arrays, output path, and statistics
    """
    valid = [f for f in feature_set if f in df_source.columns and f in df_target.columns]
    if not valid:
        print(f"  [!] No valid features for '{label}', skipping.")
        return None

    target_path  = df_target.iloc[0]['path']
    target_audio = estd.MonoLoader(filename=target_path)()
    generated    = np.zeros(len(target_audio))
    chosen_ids   = []

    # Fit KNN once on the full source collection
    k = min(n_random, len(df_source))
    nbrs = NearestNeighbors(n_neighbors=k, algorithm='ball_tree')
    nbrs.fit(df_source[valid].values)

    print(f"  Reconstructing '{label}' | features: {valid[:3]}{'…' if len(valid)>3 else ''} | n_random={n_random}")

    for i in range(len(df_target)):
        if (i + 1) % 100 == 0:
            print(f"    frame {i+1}/{len(df_target)}", end='\r')

        target_frame = df_target.iloc[i]
        query = target_frame[valid].values.reshape(1, -1)
        _, indices = nbrs.kneighbors(query)

        idx = indices[0][0] if n_random == 1 else indices[0][np.random.randint(0, len(indices[0]))]
        src_frame = df_source.iloc[idx]
        chosen_ids.append(src_frame['freesound_id'])

        n_samp      = int(target_frame['end_sample']) - int(target_frame['start_sample'])
        frame_audio = get_audio_file_segment(src_frame['path'], src_frame['start_sample'], n_samp)
        start       = int(target_frame['start_sample'])
        generated[start:start + len(frame_audio)] = frame_audio

    out_path = f'reconstructed_{label}.wav'
    estd.MonoWriter(filename=out_path, format='wav', sampleRate=44100)(essentia.array(generated))

    n_unique = len(set(chosen_ids))
    print(f"    Done → {out_path}  |  {n_unique} unique source sounds used")

    return {
        'label'         : label,
        'audio'         : generated,
        'target_audio'  : target_audio,
        'target_path'   : target_path,
        'filename'      : out_path,
        'n_unique'      : n_unique,
        'chosen_ids'    : chosen_ids,
        'features'      : valid,
        'n_random'      : n_random,
    }


def plot_reconstruction(result, save=True):
    """Plot original vs reconstructed waveforms with audio players."""
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=True)
    fig.patch.set_facecolor('white')

    orig = result['target_audio']
    gen  = result['audio']

    ax1.plot(orig, color=C['deep_purple'], linewidth=0.4, alpha=0.9)
    ax1.set_ylim(-1.05, 1.05)
    ax1.set_ylabel('Amplitude')
    ax1.set_title('Original Target', fontweight='bold', color=C['dark_grey'])

    ax2.plot(gen, color=C['mid_purple'], linewidth=0.4, alpha=0.9)
    ax2.set_ylim(-1.05, 1.05)
    ax2.set_ylabel('Amplitude')
    ax2.set_xlabel('Sample')
    ax2.set_title(
        f'Reconstructed — {result["label"]}  |  {result["n_unique"]} unique source sounds',
        fontweight='bold', color=C['dark_grey']
    )

    plt.tight_layout()
    if save:
        path = f'plot_{result["label"]}.png'
        plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()

    print('▶ Original')
    display(Audio(orig, rate=44100))
    print('▶ Reconstructed')
    display(Audio(gen, rate=44100))
    print('▶ Mix (50/50)')
    display(Audio(gen * 0.5 + orig * 0.5, rate=44100))

## Experiment 1 — Baseline: MFCCs Only

Reconstruct both target files using only the 13 MFCC coefficients as similarity features.
This is the default approach and serves as the reference point for all other experiments.

**Feature set:** `mfcc_0` … `mfcc_12`

In [ ]:
MFCC_FEATURES = [f'mfcc_{i}' for i in range(13)]

# Load both target DataFrames
df_kick = pd.read_csv('dataframe_target_kick.csv', index_col=0)
df_acid = pd.read_csv('dataframe_target_acid.csv', index_col=0)

print("=== BASELINE: Kick loop ===")
r_kick_baseline = reconstruct(df_kick, df_source, MFCC_FEATURES, n_random=1, label='kick_baseline')
plot_reconstruction(r_kick_baseline)

print("\n=== BASELINE: Acid bass loop ===")
r_acid_baseline = reconstruct(df_acid, df_source, MFCC_FEATURES, n_random=1, label='acid_baseline')
plot_reconstruction(r_acid_baseline)

## Experiment 2 — Feature Set Comparison

We test 4 different feature combinations on the kick loop to understand what each feature contributes to the similarity search.

| ID | Features used | What it captures |
|---|---|---|
| `feat_mfcc` | MFCCs only | Timbre (baseline) |
| `feat_mfcc_centroid` | MFCCs + spectral centroid | Timbre + brightness |
| `feat_mfcc_energy` | MFCCs + loudness + rms | Timbre + energy |
| `feat_all` | MFCCs + centroid + loudness + rms + zcr | Full feature set |

In [ ]:
feature_experiments = {
    'feat_mfcc'          : MFCC_FEATURES,
    'feat_mfcc_centroid' : MFCC_FEATURES + ['spectral_centroid'],
    'feat_mfcc_energy'   : MFCC_FEATURES + ['loudness', 'rms'],
    'feat_all'           : MFCC_FEATURES + ['spectral_centroid', 'loudness', 'rms', 'zero_crossing_rate'],
}

feat_results = {}
for label, feats in feature_experiments.items():
    print(f"\n=== {label} ===")
    feat_results[label] = reconstruct(df_kick, df_source, feats, n_random=1, label=label)
    plot_reconstruction(feat_results[label])

In [ ]:
# Comparison: unique sounds used per feature set
labels  = list(feat_results.keys())
uniques = [feat_results[l]['n_unique'] for l in labels]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(labels, uniques, color=C['exp'][:len(labels)], width=0.5, edgecolor='white')
ax.bar_label(bars, padding=4, fontweight='bold', color=C['dark_grey'])
ax.set_ylim(0, max(uniques) * 1.2)
ax.set_ylabel('Unique source sounds used')
ax.set_title('Feature Set Comparison — Source Variety per Reconstruction', fontweight='bold')
ax.set_xticklabels(labels, rotation=15, ha='right')
plt.tight_layout()
plt.savefig('exp2_feature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 3 — Randomization

Instead of always picking the **best** match (deterministic), we pick randomly from the top-N nearest neighbors.

| ID | n_random | Behavior |
|---|---|---|
| `rand_1` | 1 | Always best match — most accurate, can be repetitive |
| `rand_3` | 3 | Slight variation — good balance |
| `rand_10` | 10 | High variation — more creative, less precise |

A higher `n_random` means more variety in the source sounds used, but the frames are less spectrally accurate.

In [ ]:
random.seed(42)  # Reproducibility
np.random.seed(42)

rand_experiments = {
    'rand_1'  : 1,
    'rand_3'  : 3,
    'rand_10' : 10,
}

rand_results = {}
for label, n in rand_experiments.items():
    print(f"\n=== {label} (n_random={n}) ===")
    rand_results[label] = reconstruct(df_kick, df_source, MFCC_FEATURES, n_random=n, label=label)
    plot_reconstruction(rand_results[label])

In [ ]:
# Comparison: unique sounds used per randomization level
labels  = list(rand_results.keys())
uniques = [rand_results[l]['n_unique'] for l in labels]
n_vals  = [rand_results[l]['n_random'] for l in labels]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(labels, uniques, color=[C['deep_purple'], C['mid_purple'], C['light_purple']],
              width=0.4, edgecolor='white')
ax.bar_label(bars, padding=4, fontweight='bold', color=C['dark_grey'])
ax.set_ylim(0, max(uniques) * 1.25)
ax.set_ylabel('Unique source sounds used')
ax.set_title('Randomization — Source Variety vs Determinism', fontweight='bold')
for bar, n in zip(bars, n_vals):
    ax.text(bar.get_x() + bar.get_width()/2, 2, f'top-{n}', ha='center', color='white', fontweight='bold')
plt.tight_layout()
plt.savefig('exp3_randomization.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 4 — Frame Size Comparison

We use the three kick loop CSVs generated in Notebook 2 with different frame sizes.
Each frame size determines the **grain** of the reconstruction:

| Frame size | Duration | Effect on mosaicing |
|---|---|---|
| 2048 | ~46ms | Fine grain — captures transients, can sound choppy |
| 4096 | ~93ms | Default balance |
| 8192 | ~185ms | Coarse grain — smoother but less detail |

In [ ]:
frame_size_experiments = {
    'fs_2048' : ('dataframe_target_kick_fs2048.csv', 2048),
    'fs_4096' : ('dataframe_target_kick_fs4096.csv', 4096),
    'fs_8192' : ('dataframe_target_kick_fs8192.csv', 8192),
}

fs_results = {}
for label, (csv_path, fs) in frame_size_experiments.items():
    print(f"\n=== {label} (frame_size={fs}) ===")
    df_t = pd.read_csv(csv_path, index_col=0)
    fs_results[label] = reconstruct(df_t, df_source, MFCC_FEATURES, n_random=1, label=label)
    plot_reconstruction(fs_results[label])

In [ ]:
# Overlay waveforms for visual grain comparison
fig, axes = plt.subplots(3, 1, figsize=(15, 9), sharex=True)

for ax, (label, result), color in zip(axes, fs_results.items(), C['exp']):
    ax.plot(result['audio'], color=color, linewidth=0.4, alpha=0.9)
    fs = frame_size_experiments[label][1]
    ax.set_title(f'frame_size = {fs} samples ({fs/44100*1000:.0f}ms)  |  {result["n_unique"]} unique source sounds',
                 fontweight='bold', color=C['dark_grey'])
    ax.set_ylim(-1.05, 1.05)
    ax.set_ylabel('Amplitude')

axes[-1].set_xlabel('Sample')
plt.suptitle('Frame Size Comparison — Reconstructed Kick Loop', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp4_frame_sizes.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 5 — Beat-Synchronous Segmentation

Using the beat-aligned CSV from Notebook 2, each frame corresponds to one beat interval.
For percussive material this should produce a more rhythmically coherent reconstruction
because the frame boundaries align with the musical structure.

In [ ]:
print("=== Beat-sync segmentation ===")
df_beat = pd.read_csv('dataframe_target_kick_beatsync.csv', index_col=0)
r_beat = reconstruct(df_beat, df_source, MFCC_FEATURES, n_random=1, label='beat_sync')
plot_reconstruction(r_beat)

# Side-by-side vs fixed-frame baseline
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 6), sharex=False)

ax1.plot(r_kick_baseline['audio'], color=C['mid_purple'], linewidth=0.4)
ax1.set_title(f'Fixed frames (4096 / 93ms)  |  {r_kick_baseline["n_unique"]} unique sounds', fontweight='bold')
ax1.set_ylim(-1.05, 1.05); ax1.set_ylabel('Amplitude')

ax2.plot(r_beat['audio'], color=C['accent'], linewidth=0.4)
ax2.set_title(f'Beat-sync frames  |  {r_beat["n_unique"]} unique sounds', fontweight='bold')
ax2.set_ylim(-1.05, 1.05); ax2.set_ylabel('Amplitude'); ax2.set_xlabel('Sample')

plt.suptitle('Fixed Frame vs Beat-Synchronous Reconstruction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('exp5_beatsync.png', dpi=150, bbox_inches='tight')
plt.show()

## Experiment 6 — Source Collection Comparison: Piano vs Dark Techno

We use the **same target** (kick loop) with **two different source collections**:
- `dataframe_source.csv` — dark techno / acid sounds (63 sounds)
- `dataframe_source_piano.csv` — piano notes and chords (~60 sounds)

This is one of the most interesting comparisons for the report: the same target reconstructed
with sonically opposite materials produces very different textures, even though the KNN
similarity search uses the same feature set.

In [ ]:
if os.path.exists('dataframe_source_piano.csv'):
    df_source_piano = pd.read_csv('dataframe_source_piano.csv', index_col=0)
    print(f"Piano source: {len(df_source_piano)} frames")

    print("\n=== Piano source — Kick target ===")
    r_piano_kick = reconstruct(df_kick, df_source_piano, MFCC_FEATURES, n_random=1, label='piano_kick')
    plot_reconstruction(r_piano_kick)

    # Side-by-side comparison
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7), sharex=False)

    ax1.plot(r_kick_baseline['audio'], color=C['mid_purple'], linewidth=0.4)
    ax1.set_ylim(-1.05, 1.05)
    ax1.set_title(f'Source: Dark Techno  |  {r_kick_baseline["n_unique"]} unique sounds', fontweight='bold')
    ax1.set_ylabel('Amplitude')

    ax2.plot(r_piano_kick['audio'], color=C['accent'], linewidth=0.4)
    ax2.set_ylim(-1.05, 1.05)
    ax2.set_title(f'Source: Piano  |  {r_piano_kick["n_unique"]} unique sounds', fontweight='bold')
    ax2.set_ylabel('Amplitude')
    ax2.set_xlabel('Sample')

    plt.suptitle('Source Collection Comparison — Kick Loop Target', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('exp6_piano_vs_techno.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("\n=== Piano source — Acid target ===")
    r_piano_acid = reconstruct(df_acid, df_source_piano, MFCC_FEATURES, n_random=1, label='piano_acid')
    plot_reconstruction(r_piano_acid)
else:
    print("[!] dataframe_source_piano.csv not found. Run Notebook 1 + 2 first.")
    r_piano_kick = None
    r_piano_acid = None

## Experiment 7 — Tonality-Based Reconstruction

Instead of using only spectral features (MFCCs) for similarity, we first **filter the source collection
by matching tonality** (key + scale) before running KNN.

The idea: frames in the target and source that share the same musical key are more likely
to blend well tonally, even if their timbral features are not identical.

**Pipeline:**
1. For each target frame, get its detected key and scale
2. Filter source frames to those with the same key + scale (fallback to full collection if < 5 matches)
3. Run KNN on the filtered subset using MFCCs

**Note:** key detection on short frames (~93ms) is noisy by nature. The `key_strength` value
indicates reliability — frames with low key_strength are filtered out first.

In [ ]:
def reconstruct_with_tonality(df_target, df_source, label='tonality'):
    """
    Reconstruct target using tonality pre-filtering + MFCC KNN.
    Frames with key_strength < 0.3 skip tonality filtering (unreliable key detection).
    """
    has_tonality = ('key' in df_source.columns and 'key' in df_target.columns)
    if not has_tonality:
        print("[!] Tonality features not found. Re-run Notebook 2 with the updated analyze_sound().")
        return None

    target_path  = df_target.iloc[0]['path']
    target_audio = estd.MonoLoader(filename=target_path)()
    generated    = np.zeros(len(target_audio))
    chosen_ids   = []
    tonality_hits   = 0  # frames where tonality filter was applied
    tonality_misses = 0  # frames where fallback to full collection was used

    print(f"  Reconstructing '{label}' with tonality pre-filtering…")

    for i in range(len(df_target)):
        if (i + 1) % 100 == 0:
            print(f"    frame {i+1}/{len(df_target)}", end='\r')

        target_frame  = df_target.iloc[i]
        target_key    = target_frame.get('key', 'unknown')
        target_scale  = target_frame.get('scale', 'unknown')
        key_strength  = target_frame.get('key_strength', 0.0)

        # Only apply tonality filter when key detection is reliable
        if key_strength >= 0.3 and target_key != 'unknown':
            mask       = (df_source['key'] == target_key) & (df_source['scale'] == target_scale)
            df_subset  = df_source[mask]
            if len(df_subset) >= 5:
                tonality_hits += 1
            else:
                df_subset     = df_source
                tonality_misses += 1
        else:
            df_subset     = df_source
            tonality_misses += 1

        valid = [f for f in MFCC_FEATURES if f in df_subset.columns]
        query = target_frame[valid].values.reshape(1, -1)

        nbrs = NearestNeighbors(n_neighbors=1, algorithm='ball_tree')
        nbrs.fit(df_subset[valid].values)
        _, indices = nbrs.kneighbors(query)

        src_frame  = df_subset.iloc[indices[0][0]]
        chosen_ids.append(src_frame['freesound_id'])

        n_samp      = int(target_frame['end_sample']) - int(target_frame['start_sample'])
        frame_audio = get_audio_file_segment(src_frame['path'], src_frame['start_sample'], n_samp)
        start       = int(target_frame['start_sample'])
        generated[start:start + len(frame_audio)] = frame_audio

    out_path = f'reconstructed_{label}.wav'
    estd.MonoWriter(filename=out_path, format='wav', sampleRate=44100)(essentia.array(generated))

    n_unique = len(set(chosen_ids))
    hit_pct  = tonality_hits / (tonality_hits + tonality_misses) * 100
    print(f"    Done → {out_path}")
    print(f"    Tonality filter applied: {tonality_hits} frames ({hit_pct:.1f}%) | fallback: {tonality_misses} frames")
    print(f"    Unique source sounds used: {n_unique}")

    return {
        'label'        : label,
        'audio'        : generated,
        'target_audio' : target_audio,
        'target_path'  : target_path,
        'filename'     : out_path,
        'n_unique'     : n_unique,
        'chosen_ids'   : chosen_ids,
        'features'     : MFCC_FEATURES,
        'n_random'     : 1,
        'tonality_hits': tonality_hits,
        'tonality_miss': tonality_misses,
    }

print("reconstruct_with_tonality() defined")

In [ ]:
# Run tonality experiment — kick target, techno source
r_tonality = reconstruct_with_tonality(df_kick, df_source, label='tonality_kick')

if r_tonality:
    plot_reconstruction(r_tonality)

    # Compare baseline vs tonality
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 7))

    ax1.plot(r_kick_baseline['audio'], color=C['mid_purple'], linewidth=0.4)
    ax1.set_ylim(-1.05, 1.05)
    ax1.set_title(f'Baseline (MFCCs only)  |  {r_kick_baseline["n_unique"]} unique sounds', fontweight='bold')
    ax1.set_ylabel('Amplitude')

    ax2.plot(r_tonality['audio'], color=C['deep_purple'], linewidth=0.4)
    ax2.set_ylim(-1.05, 1.05)
    hits = r_tonality['tonality_hits']
    total = hits + r_tonality['tonality_miss']
    ax2.set_title(
        f'Tonality pre-filter  |  {r_tonality["n_unique"]} unique sounds  |  filter applied {hits}/{total} frames',
        fontweight='bold'
    )
    ax2.set_ylabel('Amplitude')
    ax2.set_xlabel('Sample')

    plt.suptitle('MFCCs Baseline vs Tonality Pre-filtering — Kick Loop', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('exp7_tonality.png', dpi=150, bbox_inches='tight')
    plt.show()

## 9. Summary

Overview of all reconstructions and the number of unique source sounds used in each.
A higher number indicates more variety — the mosaicing is drawing from a wider range of the source collection.

In [ ]:
all_results = {
    'kick_baseline' : r_kick_baseline,
    'acid_baseline' : r_acid_baseline,
    **feat_results,
    **rand_results,
    **fs_results,
    'beat_sync'     : r_beat,
    'piano_kick'    : r_piano_kick if 'r_piano_kick' in dir() else None,
    'piano_acid'    : r_piano_acid if 'r_piano_acid' in dir() else None,
    'tonality_kick' : r_tonality   if 'r_tonality'   in dir() else None,
}

# Build summary table
rows = []
for label, r in all_results.items():
    if r:
        rows.append({
            'Experiment'     : label,
            'Target'         : 'acid' if 'acid' in label else 'kick',
            'Features'       : ', '.join(r['features'][:3]) + ('…' if len(r['features'])>3 else ''),
            'n_random'       : r['n_random'],
            'Unique sounds'  : r['n_unique'],
            'Output file'    : r['filename'],
        })

df_summary = pd.DataFrame(rows)
display(df_summary)
df_summary.to_csv('reconstruction_summary.csv', index=False)
print("Saved: reconstruction_summary.csv")

In [ ]:
# Bar chart — all experiments
fig, ax = plt.subplots(figsize=(16, 6))
labels  = df_summary['Experiment'].tolist()
uniques = df_summary['Unique sounds'].tolist()

colors = []
for label in labels:
    if 'acid' in label:   colors.append(C['mid_grey'])
    elif 'rand' in label: colors.append(C['accent'])
    elif 'feat' in label: colors.append(C['mid_purple'])
    elif 'fs'   in label: colors.append(C['light_purple'])
    elif 'beat' in label: colors.append(C['deep_purple'])
    else:                 colors.append(C['dark_grey'])

bars = ax.bar(labels, uniques, color=colors, edgecolor='white', width=0.6)
ax.bar_label(bars, padding=3, fontweight='bold', color=C['dark_grey'])
ax.set_ylim(0, max(uniques) * 1.2)
ax.set_ylabel('Unique source sounds used')
ax.set_title('All Experiments — Source Sound Variety', fontsize=13, fontweight='bold')
ax.set_xticklabels(labels, rotation=25, ha='right', fontsize=9)

from matplotlib.patches import Patch
legend_items = [
    Patch(color=C['dark_grey'],   label='Baseline'),
    Patch(color=C['mid_grey'],    label='Acid target'),
    Patch(color=C['mid_purple'],  label='Feature set'),
    Patch(color=C['accent'],      label='Randomization'),
    Patch(color=C['light_purple'],label='Frame size'),
    Patch(color=C['deep_purple'], label='Beat-sync'),
]
ax.legend(handles=legend_items, loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('exp_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: exp_summary.png")

In [ ]:
# Download all outputs
from google.colab import files
import os

to_download = (
    [r['filename'] for r in all_results.values() if r] +
    [f for f in os.listdir('.') if f.endswith('.png') and f.startswith('exp')] +
    ['reconstruction_summary.csv']
)

print("Files ready to download:")
for f in to_download:
    if os.path.exists(f):
        print(f"  ✓ {f}")
    else:
        print(f"  ✗ MISSING: {f}")